# RAG (검색 증강 생성)

**RAG(Retrieval-Augmented Generation)**는 대규모 언어 모델(LLM)이 **외부의 지식(데이터)**을 참고하여 더 정확하고 풍부한 답변을 생성하도록 만드는 기술입니다.

쉽게 비유하자면, **"시험을 볼 때 교과서를 펼쳐 놓고 답을 찾는 오픈 북 테스트(Open-book Test)"**와 같습니다.

---

### **왜 RAG가 필요한가요?**
LLM(ChatGPT 등)은 다음과 같은 한계:
1.  **최신 정보 부재:** 학습 시점 이후의 정보는 알지 못합니다.
2.  **환각 현상 (Hallucination):** 모르는 내용도 사실인 것처럼 거짓말을 할 수 있습니다.
3.  **사내/비공개 데이터 접근 불가:** 기업 내부 문서나 개인적인 자료는 학습하지 않았습니다.

**👉 RAG는 외부 데이터베이스에서 "정답의 근거"를 찾아와서 LLM에게 제공함으로써 이 문제들을 해결합니다.**

---

### **RAG의 작동 원리 (3단계)**

1.  **검색 (Retrieval):** 사용자의 질문과 관련된 문서를 벡터 DB(Vector DB) 등에서 찾아옵니다.
2.  **증강 (Augmented):** 찾아온 정보를 프롬프트에 끼워 넣어(Context) 질문을 보강합니다.
3.  **생성 (Generation):** LLM은 보강된 정보를 바탕으로 신뢰할 수 있는 답변을 생성합니다.



In [ ]:
# RAG(검색 증강 생성) 시스템의 맛보기 예제
# 이 코드는 RAG의 기본 개념을 실습하기 위한 간단한 예시입니다

# 필요한 라이브러리 설치
# LangChain은 LLM 애플리케이션 구축을 위한 프레임워크로,
# 다양한 LLM 모델과 도구들을 통합하여 사용할 수 있게 해줍니다
!pip install langchain_openai langchain_core

# 코드 실행 흐름 설명:
# 1. 라이브러리 설치 및 임포트 → 2. API 키 설정 → 3. LLM 모델 초기화 →
# 4. 컨텍스트(지식) 정의 → 5. 프롬프트 템플릿 생성 → 6. 처리 파이프라인 구성 → 7. 실행

# 이 코드는 "오픈 북 테스트" 개념을 구현한 것입니다:
# - 컨텍스트(context): 교과서 역할 (우리가 제공하는 지식)
# - LLM: 시험 보는 학생 (컨텍스트를 참고하여 답변 생성)
# - 프롬프트: 시험 문제지 (질문과 함께 컨텍스트를 제공)

# 왜 이런 방식이 필요한가?
# 1. LLM은 학습 데이터 이후의 정보나 비공개 데이터를 알지 못함
# 2. LLM은 때로 환각(hallucination) 현상으로 잘못된 정보를 생성할 수 있음
# 3. RAG는 외부 지식 소스를 제공함으로써 정확하고 최신의 답변을 생성하도록 도와줌

# 기계적 구동 과정:
# 1. 사용자 질문이 입력되면, 컨텍스트와 함께 프롬프트에 삽입됨
# 2. 프롬프트 템플릿이 완성된 프롬프트 문자열로 변환됨
# 3. 이 프롬프트가 LLM 모델로 전달되어 처리됨
# 4. LLM의 응답이 출력 파서를 통해 문자열로 변환됨
# 5. 최종 답변이 사용자에게 반환됨

# LangChain의 역할:
# - 다양한 LLM 제공업체들을 통합한 표준 인터페이스 제공
# - 모듈화된 컴포넌트들(프롬프트, 체인, 메모리 등)을 조립하기 쉬움
# - RAG 시스템 구축에 필요한 도구들을 제공

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 2.9 MB/s eta 0:00:00


In [ ]:
# 라이브러리 임포트
import os
from langchain_openai import ChatOpenAI  # OpenAI의 채팅 모델 사용을 위한 클래스
from langchain_core.prompts import PromptTemplate  # 프롬프트 템플릿 관리 클래스
from langchain_core.output_parsers import StrOutputParser  # LLM 출력을 문자열로 변환하는 파서

LangChain 없이 직접 모델을 바꾸려면, 모델마다 API 사용 방식이 달라서 각각의 문서를 따로 찾아봐야 한다는 어려움이 있습니다.

예를 들어:

- Gemini API: https://ai.google.dev/gemini-api/docs?hl=ko
- OpenAI API: https://platform.openai.com/docs/guides/text”


In [ ]:
# API 키 설정
import getpass
import os

# OpenAI API 키를 안전하게 입력받아 환경변수에 저장
# getpass.getpass()는 터미널에서 입력값을 안보이게 해줌 (Colab에서는 별도 창이 뜸)
# 이 키는 OpenAI API를 사용하기 위한 인증 정보로, 모든 API 호출에 필요합니다
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

# 코드 실행 흐름 설명:
# 1. 라이브러리 설치 및 임포트 → 2. API 키 설정 → 3. LLM 모델 초기화 →
# 4. 컨텍스트(지식) 정의 → 5. 프롬프트 템플릿 생성 → 6. 처리 파이프라인 구성 → 7. 실행

# 이 코드는 "오픈 북 테스트" 개념을 구현한 것입니다:
# - 컨텍스트(context): 교과서 역할 (우리가 제공하는 지식)
# - LLM: 시험 보는 학생 (컨텍스트를 참고하여 답변 생성)
# - 프롬프트: 시험 문제지 (질문과 함께 컨텍스트를 제공)

# 왜 이런 방식이 필요한가?
# 1. LLM은 학습 데이터 이후의 정보나 비공개 데이터를 알지 못함
# 2. LLM은 때로 환각(hallucination) 현상으로 잘못된 정보를 생성할 수 있음
# 3. RAG는 외부 지식 소스를 제공함으로써 정확하고 최신의 답변을 생성하도록 도와줌

# 기계적 구동 과정:
# 1. 사용자 질문이 입력되면, 컨텍스트와 함께 프롬프트에 삽입됨
# 2. 프롬프트 템플릿이 완성된 프롬프트 문자열로 변환됨
# 3. 이 프롬프트가 LLM 모델로 전달되어 처리됨
# 4. LLM의 응답이 출력 파서를 통해 문자열로 변환됨
# 5. 최종 답변이 사용자에게 반환됨

# LangChain의 역할:
# - 다양한 LLM 제공업체들을 통합한 표준 인터페이스 제공
# - 모듈화된 컴포넌트들(프롬프트, 체인, 메모리 등)을 조립하기 쉬움
# - RAG 시스템 구축에 필요한 도구들을 제공

Enter your OpenAI API key: ··········


In [ ]:
# 1. 모델 준비
# ChatOpenAI 클래스를 사용하여 OpenAI의 채팅 모델을 초기화합니다
# model="gpt-4o-mini": 사용할 모델의 이름 (GPT-4o-mini는 가성비 좋은 최신 모델)
# temperature=0: 모델의 창의성을 조절 (0 = 결정적 답변, 높을수록 창의적)
# temperature가 0이면 동일한 입력에 대해 항상 같은 출력을 생성함
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. 우리 가게만의 지식 (Context) - 아직 DB 없이 변수로 직접 정의
# RAG의 핵심: 외부 지식 소스를 모델에게 제공하는 컨텍스트
# 실제 RAG 시스템에서는 이 컨텍스트가 벡터 데이터베이스에서 동적으로 검색되지만,
# 지금은 간단한 예제를 위해 문자열 변수로 하드코딩합니다
context = """
[맛있는 김밥집 메뉴판]
1. 원조 김밥: 3,000원 - 40년 전통의 맛
2. 치즈 폭탄 김밥: 4,500원 - 모짜렐라 치즈가 듬뿍
3. 불닭 김밥: 5,000원 - 아주 매움, 맵찔이 금지
* 영업 시간: 오전 10시 ~ 오후 8시 (매주 월요일 휴무)
"""

# 이 부분의 RAG에서의 역할:
# 컨텍스트는 LLM이 참고할 수 있는 "지식베이스" 또는 "참고서"입니다
# 실제 RAG 시스템에서는 사용자 질문과 관련된 문서 조각들을 검색하여 동적으로 구성됩니다

# 기계적 구동 과정에서 컨텍스트의 역할:
# 1. 사용자의 질문이 들어오면 (나중 단계에서)
# 2. 컨텍스트 전체가 프롬프트 템플릿의 특정 위치({context})에 삽입됩니다
# 3. LLM은 "이 정보를 참고해서 답변하라"는 지시와 함께 컨텍스트를 받게 됩니다
# 4. LLM은 컨텍스트 안에서 관련 정보를 찾아 답변을 생성합니다

# 왜 컨텍스트가 필요한가?
# 1. LLM 자체는 이 김밥집의 메뉴 정보를 학습하지 않았습니다 (비공개 데이터)
# 2. 컨텍스트를 제공함으로써 LLM은 자신이 학습하지 않은 정보도 참고할 수 있습니다
# 3. 이는 "오픈북 시험"에서 교과서를 제공하는 것과 같은 원리입니다

# 실제 RAG 시스템과의 차이점:
# - 실제: 컨텍스트는 벡터 DB에서 질문과 유사한 문서를 검색하여 동적으로 구성
# - 현재: 모든 컨텍스트를 항상 전체 제공 (효율적이지 않지만 개념 이해용)

# 컨텍스트 설계 시 고려사항:
# 1. 관련성: 질문과 관련된 정보만 포함해야 함
# 2. 정확성: 제공되는 정보가 정확해야 함
# 3. 구조화: 정보가 잘 정리되어 있어 LLM이 이해하기 쉬워야 함

In [ ]:
# 3. 프롬프트 템플릿 만들기 (상세 분석)
# 이 부분은 RAG 시스템의 "사용 설명서"를 작성하는 단계입니다
# LLM이 어떻게 컨텍스트를 사용하고, 어떤 방식으로 답변해야 하는지 구체적으로 지시합니다

# RAG의 핵심: {context} 자리에 검색된 문서를 넣는 것!
# {context}는 실제 실행 시점에 외부에서 제공된 지식으로 채워집니다
template = """
당신은 친절한 김밥집 점원입니다.  # 1. 역할 설정 (페르소나 정의)
아래 [참고 정보]를 바탕으로 손님의 [질문]에 답변해주세요.  # 2. 컨텍스트 사용 지시
정보에 없는 내용은 "잘 모르겠습니다"라고 답하세요.  # 3. 환각 방지 제한 조건

[참고 정보]  # 4. 컨텍스트 영역 표시 (구조화)
{context}  # 5. 외부 지식 삽입 지점 (동적 콘텐츠)

[질문]: {question}  # 6. 사용자 질문 삽입 지점
"""

# PromptTemplate.from_template() 메서드의 내부 작동:
# 1. 템플릿 문자열을 분석하여 {context}, {question} 같은 변수(자리표시자)를 감지
# 2. 이 변수들을 "input_variables" 리스트에 저장
# 3. 템플릿 객체를 생성하여 나중에 변수 값을 채울 수 있는 구조 준비
prompt = PromptTemplate.from_template(template)

# 프롬프트 템플릿이 RAG 파이프라인에서 처리되는 구체적 과정:
#
# [템플릿 준비 단계]
# template 문자열 → PromptTemplate 객체 변환
# 객체 속성: template="...", input_variables=['context', 'question']
#
# [실행 시점 (invoke 메서드 호출 시)]
# 1. prompt.format(context=실제_컨텍스트, question=실제_질문) 호출
# 2. format() 메서드가 {context}를 실제_컨텍스트 문자열로 치환
# 3. {question}을 실제_질문 문자열로 치환
# 4. 최종 프롬프트 예시:
#    """
#    당신은 친절한 김밥집 점원입니다.
#    아래 [참고 정보]를 바탕으로 손님의 [질문]에 답변해주세요.
#    정보에 없는 내용은 "잘 모르겠습니다"라고 답하세요.
#
#    [참고 정보]
#    [맛있는 김밥집 메뉴판]
#    1. 원조 김밥: 3,000원 - 40년 전통의 맛
#    2. 치즈 폭탄 김밥: 4,500원 - 모짜렐라 치즈가 듬뿍
#    3. 불닭 김밥: 5,000원 - 아주 매움, 맵찔이 금지
#    * 영업 시간: 오전 10시 ~ 오후 8시 (매주 월요일 휴무)
#
#    [질문]: 불닭 김밥 얼마인가요?
#    """
# 5. 이 완성된 프롬프트가 LLM으로 전송됨

# 프롬프트 템플릿 설계의 심리학적 원리:
# 1. 역할 부여 효과: "김밥집 점원"이라는 역할을 부여함으로써 LLM의 답변 스타일과 톤을 조정
# 2. 구체성의 힘: "아래 [참고 정보]를 바탕으로"라는 명확한 지시는 LLM이 컨텍스트를 무시하고
#    사전 학습된 지식만으로 답변하는 것을 방지
# 3. 안전 장치: "정보에 없는 내용은..." 구문은 LLM의 과도한 확신을 제한

# 실제 RAG 시스템에서의 프롬프트 최적화 기술:
# 1. Few-shot 프롬프팅: 템플릿에 예시 답변을 포함시켜 LLM의 답변 품질 향상
# 2. 체이닝: 여러 프롬프트를 순차적으로 연결하여 복잡한 작업 처리
# 3. 조건부 프롬프팅: 질문 유형에 따라 다른 템플릿 선택
# 4. 컨텍스트 압축: 긴 컨텍스트를 요약하거나 가장 관련성 높은 부분만 추출

# 이 템플릿의 한계와 개선 방향:
# 1. 컨텍스트가 길 경우 LLM의 토큰 제한을 초과할 수 있음 → 청킹(chunking) 필요
# 2. 모든 컨텍스트를 무조건 포함 → 관련성 없는 정보도 포함될 수 있음 → 검색 품질 개선 필요
# 3. 고정된 답변 형식 → 더 유연한 응답이 필요할 수 있음

# LangChain의 PromptTemplate 고급 기능:
# - 부분 변수(partial variables): 일부 변수를 미리 채워놓을 수 있음
# - 템플릿 구성요소 조합: 여러 템플릿을 결합하여 사용 가능
# - 출력 파서와 연동: LLM 출력을 특정 구조(JSON 등)로 파싱할 수 있도록 지시 포함 가능

# 이 프롬프트 템플릿이 RAG의 "증강(Augmented)" 단계에서 수행하는 역할:
# 검색된 문서(Retrieval) + 사용자 질문 → 프롬프트 템플릿으로 증강 → LLM에 입력

In [ ]:
# 4. LangChain의 파이프라인: LangChain Expression Language(LCEL)
# LCEL은 LangChain의 선언적 API로, 구성요소들을 파이프라인으로 연결하는 직관적인 방법을 제공합니다
# 여기서는 "프롬프트 → LLM → 문자열 변환"의 3단계 파이프라인을 구성합니다

# 파이프(|) 연산자를 사용하여 구성요소들을 연결합니다
# 이는 Unix의 파이프라인 개념과 유사하게, 데이터가 왼쪽에서 오른쪽으로 흐르도록 설계되었습니다
chain = prompt | llm | StrOutputParser()

# 각 구성요소의 역할과 데이터 변환 과정:
#
# [prompt] 단계:
# - 입력: {"context": 컨텍스트_문자열, "question": 질문_문자열} 형태의 딕셔너리
# - 처리: PromptTemplate의 format() 메서드가 호출되어 템플릿의 자리표시자를 실제 값으로 대체
# - 출력: 완성된 프롬프트 문자열 (컨텍스트와 질문이 삽입된)
#
# [llm] 단계:
# - 입력: prompt 단계에서 출력된 완성된 프롬프트 문자열
# - 처리: ChatOpenAI 객체가 OpenAI API를 호출하여 LLM 모델에 프롬프트 전송
# - 출력: ChatMessage 객체 (OpenAI의 응답 형식)
#
# [StrOutputParser] 단계:
# - 입력: llm 단계에서 출력된 ChatMessage 객체
# - 처리: ChatMessage 객체에서 텍스트 내용만 추출하여 문자열로 변환
# - 출력: 순수 문자열 형태의 최종 답변

# LCEL 파이프라인의 기계적 실행 흐름:
# 1. chain.invoke(input_dict)가 호출되면:
#    - input_dict는 {"context": "실제 컨텍스트", "question": "실제 질문"} 형태
# 2. 파이프라인의 첫 번째 요소(prompt)가 input_dict를 받아 처리
# 3. prompt의 출력이 자동으로 llm의 입력으로 전달됨
# 4. llm의 출력이 자동으로 StrOutputParser의 입력으로 전달됨
# 5. StrOutputParser의 출력이 최종 결과로 반환됨

# LCEL의 내부 동작 원리:
# - 각 구성요소는 __or__ 메서드를 구현하여 파이프(|) 연산을 지원
# - 파이프라인은 실제로 RunnableSequence 객체로 변환됨
# - invoke() 메서드가 호출될 때까지 실제 API 호출은 발생하지 않음 (지연 실행)

# 왜 LCEL을 사용하는가?
# 1. 가독성: 처리 흐름을 직관적으로 표현할 수 있음
# 2. 재사용성: 파이프라인을 하나의 객체로 만들어 여러 번 재사용 가능
# 3. 유연성: 새로운 구성요소를 쉽게 추가하거나 교체할 수 있음
# 4. 디버깅: 각 단계의 입력/출력을 쉽게 검사할 수 있음

# 실제 RAG 시스템에서의 LCEL 확장:
# 이 간단한 파이프라인은 더 복잡한 RAG 시스템의 기본 골격입니다:
#
# 1. 검색기(retriever) 추가:
#    retriever | prompt | llm | StrOutputParser()
#    (질문 → 문서 검색 → 프롬프트 구성 → LLM → 답변)
#
# 2. 메모리(memory) 추가:
#    대화 기록을 관리하여 이전 문맥을 유지
#
# 3. 다중 체인 구성:
#    여러 개의 체인을 조합하여 복잡한 작업 처리
#
# 4. 조건부 분기:
#    질문 유형에 따라 다른 처리 경로로 분기

# StrOutputParser의 구체적 역할:
# - LLM 응답은 보통 구조화된 형식(예: AIMessage, ChatMessage)으로 반환됨
# - StrOutputParser는 이 구조화된 객체에서 content 속성을 추출하여 문자열로 변환
# - 만약 LLM이 JSON 등 다른 형식으로 응답한다면, 다른 OutputParser를 사용할 수 있음

# 파이프라인 확장 예시:
# 원한다면 더 많은 단계를 추가할 수 있습니다:
# chain = prompt | llm | SomeFilter() | StrOutputParser()
# 여기서 SomeFilter()는 LLM 응답을 후처리하는 사용자 정의 구성요소입니다

# LCEL의 고급 기능:
# 1. 병렬 처리: 여러 작업을 동시에 실행
# 2. 폴백: 주요 모델 실패 시 대체 모델로 자동 전환
# 3. 로깅: 각 단계의 입력/출력을 기록
# 4. 스트리밍: 응답을 청크 단위로 실시간 반환

# 이 파이프라인이 RAG의 "생성(Generation)" 단계를 담당합니다:
# 검색된 문서로 증강된 프롬프트 → LLM을 통한 답변 생성 → 사용자 친화적 형식으로 변환

# 디버깅 팁:
# 파이프라인의 특정 단계에서 어떤 데이터가 흐르는지 확인하려면:
# - 각 단계를 개별적으로 실행해볼 수 있음
# - LangChain의 callback 기능을 사용하여 로그 기록
# - .pipe() 메서드를 사용하여 중간 결과 확인

# 참고: 이 파이프라인 구성은 함수형 프로그래밍의 합성(composition) 개념과 유사합니다
# 각 구성요소는 순수 함수처럼 동작하며, 입력을 받아 출력을 생성합니다

In [ ]:
# 5. 실행 및 결과 확인
# 이 부분은 완성된 RAG 파이프라인을 실제로 실행하고 결과를 확인하는 단계입니다

# 사용자의 질문을 정의합니다
# 이 질문은 두 부분으로 구성되어 있습니다:
# 1. 가격 정보 요청: "불닭 김밥 얼마인가요?"
# 2. 영업 시간 확인: "지금 저녁 9시인데 가도 되나요?"
question = "불닭 김밥 얼마인가요? 그리고 지금 저녁 9시인데 가도 되나요?"

# chain.invoke() 메서드를 호출하여 RAG 파이프라인 실행
# 입력: {"context": context, "question": question} 형태의 딕셔너리
# - context: 앞서 정의한 김밥집 메뉴판 정보 (문자열)
# - question: 위에서 정의한 사용자 질문 (문자열)
response = chain.invoke({"context": context, "question": question})

# 체인의 기계적 실행 과정 상세 설명:
#
# 1. chain.invoke() 호출 시 내부적으로 발생하는 일련의 과정:
#    입력 → {"context": context변수값, "question": question변수값}
#    ↓
#    [prompt 단계] PromptTemplate이 입력 딕셔너리를 받아 처리:
#    - prompt.format(context=context, question=question) 실행
#    - 템플릿의 {context} 자리에 context 문자열 삽입
#    - 템플릿의 {question} 자리에 question 문자열 삽입
#    - 출력: 완성된 프롬프트 문자열 (약 400-500자 정도)
#    ↓
#    [llm 단계] ChatOpenAI 객체가 프롬프트 문자열을 받아 처리:
#    - HTTP 요청으로 OpenAI API 엔드포인트에 프롬프트 전송
#    - 모델: gpt-4o-mini (temperature=0 설정 적용)
#    - API 응답 수신: ChatMessage 객체 (content 속성에 답변 텍스트 포함)
#    ↓
#    [StrOutputParser 단계] ChatMessage 객체를 문자열로 변환:
#    - ChatMessage 객체에서 .content 속성 값 추출
#    - 출력: 순수 문자열 "불닭 김밥은 5,000원입니다. ..."
#    ↓
#    최종 반환: 변환된 문자열이 response 변수에 저장
#
# 2. API 호출 시 실제로 전송되는 프롬프트의 예시:
#    """
#    당신은 친절한 김밥집 점원입니다.
#    아래 [참고 정보]를 바탕으로 손님의 [질문]에 답변해주세요.
#    정보에 없는 내용은 "잘 모르겠습니다"라고 답하세요.
#
#    [참고 정보]
#    [맛있는 김밥집 메뉴판]
#    1. 원조 김밥: 3,000원 - 40년 전통의 맛
#    2. 치즈 폭탄 김밥: 4,500원 - 모짜렐라 치즈가 듬뿍
#    3. 불닭 김밥: 5,000원 - 아주 매움, 맵찔이 금지
#    * 영업 시간: 오전 10시 ~ 오후 8시 (매주 월요일 휴무)
#
#    [질문]: 불닭 김밥 얼마인가요? 그리고 지금 저녁 9시인데 가도 되나요?
#    """

# 결과 출력
print("=== AI 답변 ===")
print(response)

# 예상되는 출력 결과 분석:
# 1. 불닭 김밥 가격: 컨텍스트에서 "불닭 김밥: 5,000원" 정보를 찾아 답변
# 2. 영업 시간 확인: 컨텍스트에서 "영업 시간: 오전 10시 ~ 오후 8시" 정보를 찾아 답변
# 3. 추가 정보: 월요일 휴무 정보도 포함될 수 있음
#
# 예상 답변 예시:
# "불닭 김밥은 5,000원입니다. 영업 시간은 오전 10시부터 오후 8시까지이므로,
#  지금은 저녁 9시로 영업 시간이 종료되었습니다. 따라서 지금 방문하실 수 없습니다."
#
# 또는:
# "불닭 김밥은 5,000원입니다. 저희 가게는 오전 10시부터 오후 8시까지 영업하며,
#  현재는 저녁 9시로 영업 시간이 지났기 때문에 방문하실 수 없습니다."

# 이 실행 과정이 보여주는 RAG의 핵심 가치:
# 1. **정확성**: LLM은 컨텍스트의 정확한 정보(5,000원, 영업 시간)를 참고하여 답변
# 2. **환각 방지**: 컨텍스트에 없는 정보는 "잘 모르겠습니다"라고 답변하도록 지시받음
# 3. **맥락 이해**: 두 가지 질문을 하나의 응답으로 자연스럽게 통합하여 답변
# 4. **실시간 정보 반영**: LLM의 학습 데이터 이후의 정보(이 가게의 영업 시간)도 처리 가능

# 실제 RAG 시스템과의 차이점 및 발전 방향:
# 1. **동적 검색**: 실제 시스템에서는 질문과 가장 관련 높은 문서 조각만 검색하여 컨텍스트 구성
# 2. **컨텍스트 길이 최적화**: 토큰 제한을 고려하여 가장 중요한 정보만 포함
# 3. **다중 소스 통합**: 여러 데이터베이스나 문서 저장소에서 정보 검색
# 4. **메타데이터 활용**: 문서 출처, 신뢰도 점수 등을 활용하여 답변 품질 향상

# 오류 처리 및 실패 시나리오:
# 1. API 키 문제: OpenAI API 키가 유효하지 않으면 에러 발생
# 2. 네트워크 문제: 인터넷 연결 실패 시 요청 실패
# 3. 토큰 제한 초과: 컨텍스트가 너무 길면 API 호출 실패
# 4. 모델 한계: 복잡한 추론이 필요하면 잘못된 답변 가능성

# 확장 가능성:
# 1. **대화형으로 확장**: while 루프로 사용자 입력을 계속 받아 대화 지속
# 2. **멀티턴 대화**: 이전 대화 기록을 컨텍스트에 추가하여 맥락 유지
# 3. **스트리밍 출력**: 답변을 실시간으로 청크 단위로 출력
# 4. **로그 및 모니터링**: 각 단계의 실행 시간, 토큰 사용량 등 기록

# 디버깅을 위한 팁:
# 1. 중간 결과 확인: 각 단계를 개별적으로 실행하여 출력 확인
#    예: prompt_only = prompt.invoke({"context": context, "question": question})
# 2. 토큰 수 계산: 프롬프트 길이를 확인하여 토큰 제한 초과 방지
# 3. 응답 포맷 검증: StrOutputParser 대신 다른 파서로 변경하여 구조화된 출력 확인

# 이 코드 블록의 실행 결과는 RAG 시스템의 성공 여부를 직관적으로 보여줍니다:
# - 성공: 컨텍스트의 정확한 정보를 바탕으로 한 유용한 답변
# - 실패: 컨텍스트를 무시하거나 잘못 해석한 답변
#
# 이 간단한 예제는 실제 RAG 시스템 구축의 기초가 되는 핵심 원리를 보여줍니다.

=== AI 답변 ===
불닭 김밥은 5,000원입니다. 하지만 저희 영업 시간이 오후 8시까지라서 지금은 가실 수 없습니다. 다음에 방문해 주세요!
